In [10]:
%pip install ccxt praw newsapi-python vaderSentiment pandas numpy python-dotenv

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: C:\Users\tromb\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [1]:
import os, time, datetime
import ccxt
import pandas as pd
import numpy as np
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from newsapi import NewsApiClient
import praw
from dotenv import load_dotenv
load_dotenv()

os.makedirs("data", exist_ok=True)

In [3]:
# Binance via ccxt 
BINANCE = ccxt.binance()

# Reddit (PRAW)
REDDIT_CLIENT_ID = os.getenv("REDDIT_CLIENT_ID")
REDDIT_CLIENT_SECRET = os.getenv("REDDIT_CLIENT_SECRET")
REDDIT_USER_AGENT = os.getenv("REDDIT_USER_AGENT")

# NewsAPI
NEWS_API_KEY = os.getenv("NEWS_API_KEY")

# coin list to fetch 
COINS = ["BTC/USDT", "ETH/USDT", "SOL/USDT"]

# time window for historical fetch 
HISTORY_MINUTES = 360  

# initialize clients
newsapi = NewsApiClient(api_key=NEWS_API_KEY)
reddit = praw.Reddit(
    client_id=REDDIT_CLIENT_ID,
    client_secret=REDDIT_CLIENT_SECRET,
    user_agent=REDDIT_USER_AGENT
)

analyzer = SentimentIntensityAnalyzer()

In [4]:
def fetch_ohlcv(symbol, limit=HISTORY_MINUTES):
    # Binance returns
    bars = BINANCE.fetch_ohlcv(symbol, timeframe='1m', limit=limit)
    df = pd.DataFrame(bars, columns=['timestamp_ms','open','high','low','close','volume'])
    df['timestamp'] = pd.to_datetime(df['timestamp_ms'], unit='ms')
    df = df[['timestamp','open','high','low','close','volume']]
    df['symbol'] = symbol
    return df

def fetch_reddit_count_and_sentiment(query, since_minutes=60, max_posts=200):
    """
    Search subreddit 'all' for recent posts/comments containing query.
    Returns count and average sentiment (compound) using VADER.
    """
    cutoff = datetime.datetime.utcnow() - datetime.timedelta(minutes=since_minutes)
    count = 0
    comp_scores = []
    # use subreddit search 
    try:
        for submission in reddit.subreddit('all').search(query, sort='new', limit=max_posts):
            created = datetime.datetime.utcfromtimestamp(submission.created_utc)
            if created < cutoff:
                continue
            text = (submission.title or "") + " " + (submission.selftext or "")
            s = analyzer.polarity_scores(text)
            comp_scores.append(s['compound'])
            count += 1
    except Exception:
        # fallback: zero
        return 0, 0.0
    avg_sent = float(np.mean(comp_scores)) if comp_scores else 0.0
    return count, avg_sent

def fetch_news_count_and_sentiment(query, since_minutes=60, page_size=20):
    cutoff_from = (datetime.datetime.utcnow() - datetime.timedelta(minutes=since_minutes)).strftime("%Y-%m-%dT%H:%M:%SZ")
    try:
        res = newsapi.get_everything(q=query, language='en', from_param=cutoff_from, page_size=page_size, sort_by='publishedAt')
        articles = res.get('articles', [])
        count = len(articles)
        comp_scores = []
        for a in articles:
            txt = (a.get('title') or "") + " " + (a.get('description') or "")
            s = analyzer.polarity_scores(txt)
            comp_scores.append(s['compound'])
        avg_sent = float(np.mean(comp_scores)) if comp_scores else 0.0
        return count, avg_sent
    except Exception:
        return 0, 0.0

In [5]:
rows = []
print("Fetching OHLCV for coins...")


try:
    BINANCE.enableRateLimit = True
except Exception:
    pass

def fetch_ohlcv_with_retry(symbol, limit=HISTORY_MINUTES, retries=5, delay=1.0):
    """Call existing fetch_ohlcv(...) with retries/backoff on transient errors."""
    for attempt in range(1, retries + 1):
        try:
            return fetch_ohlcv(symbol, limit=limit)
        except (ccxt.RequestTimeout, ccxt.NetworkError) as e:
            print(f"Binance timeout/network error for {symbol}, attempt {attempt}/{retries}: {e}")
            if attempt == retries:
                raise
            time.sleep(delay)
            delay *= 2
        except Exception as e:
            print(f"Error fetching OHLCV for {symbol}: {e}")
            raise

for coin in COINS:
    try:
        df_ohlcv = fetch_ohlcv_with_retry(coin, limit=HISTORY_MINUTES)
    except Exception as e:
        print(f"Failed to fetch OHLCV for {coin}, skipping. Error: {e}")
        continue

    
    base_symbol = coin.split('/')[0]
    try:
        reddit_count, reddit_sent = fetch_reddit_count_and_sentiment(base_symbol, since_minutes=60, max_posts=100)
    except Exception as e:
        print(f"Reddit fetch failed for {base_symbol}: {e}")
        reddit_count, reddit_sent = 0, 0.0

    try:
        news_count, news_sent = fetch_news_count_and_sentiment(base_symbol, since_minutes=60, page_size=20)
    except Exception as e:
        print(f"News fetch failed for {base_symbol}: {e}")
        news_count, news_sent = 0, 0.0


    for i, row in df_ohlcv.iterrows():
        ts = row['timestamp']
        rows.append({
            'timestamp': ts,
            'symbol': coin,
            'open': row['open'],
            'high': row['high'],
            'low': row['low'],
            'close': row['close'],
            'volume': row['volume'],
            'reddit_count_60m': reddit_count,
            'reddit_sent_60m': reddit_sent,
            'news_count_60m': news_count,
            'news_sent_60m': news_sent
        })
    print(f"Collected {len(df_ohlcv)} minutes for {coin}.")

#Save dataframe to CSV
df = pd.DataFrame(rows)
df = df.sort_values(['symbol','timestamp']).reset_index(drop=True)
df.to_csv("data/raw.csv", index=False)
print("Saved data/raw.csv, rows:", len(df))

Fetching OHLCV for coins...


C:\Users\tromb\AppData\Local\Temp\ipykernel_23408\2544324594.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  cutoff = datetime.datetime.utcnow() - datetime.timedelta(minutes=since_minutes)
C:\Users\tromb\AppData\Local\Temp\ipykernel_23408\2544324594.py:35: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  cutoff_from = (datetime.datetime.utcnow() - datetime.timedelta(minutes=since_minutes)).strftime("%Y-%m-%dT%H:%M:%SZ")


Collected 360 minutes for BTC/USDT.


C:\Users\tromb\AppData\Local\Temp\ipykernel_23408\2544324594.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  cutoff = datetime.datetime.utcnow() - datetime.timedelta(minutes=since_minutes)
C:\Users\tromb\AppData\Local\Temp\ipykernel_23408\2544324594.py:35: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  cutoff_from = (datetime.datetime.utcnow() - datetime.timedelta(minutes=since_minutes)).strftime("%Y-%m-%dT%H:%M:%SZ")


Collected 360 minutes for ETH/USDT.


C:\Users\tromb\AppData\Local\Temp\ipykernel_23408\2544324594.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  cutoff = datetime.datetime.utcnow() - datetime.timedelta(minutes=since_minutes)


Collected 360 minutes for SOL/USDT.
Saved data/raw.csv, rows: 1080


C:\Users\tromb\AppData\Local\Temp\ipykernel_23408\2544324594.py:35: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  cutoff_from = (datetime.datetime.utcnow() - datetime.timedelta(minutes=since_minutes)).strftime("%Y-%m-%dT%H:%M:%SZ")
